<a href="https://colab.research.google.com/github/yeonshiri/AGS/blob/main/code/ROI_mc_sa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --------------------------------------------------------------------------
# 라즈베리파이에 best.pt 파일이 있다고 가정하고 이미지 추론 후 bbox 그리고 ROI 순서대로 저장하는 코드 정리

## 1. yolov5 불러와서 실행하기(라즈베리파이에서는 한번만 실행하고 yolov5 폴더 유지하면 다시 실행 안해도 됨.)
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

## 2. 이미 사전학습된 weight best.pt 불러오기(vscode에서는 아마 ./home/pi/yolov5에 저장하게 될 것.)
from google.colab import files
import cv2
import os
import glob
import numpy as np
import shutil
import subprocess

uploaded_best = files.upload()  # 사전학습 weight인 best.pt file upload
best_filename = list(uploaded_best.keys())[0]  # best.pt의 경로를
if not os.path.exists(f"/content/{best_filename}"):
    shutil.move(best_filename, f"/content/{best_filename}")   # 항상 /content로 이용.
best_path = f"/content/{best_filename}"

# 라즈베리파이에서 경로는 가상환경 위치에 따라 다르겠지만 보통 ./home/pi/yolov5일테니까 /content만 다 얘로 바꾸면 됨.

## 3. 채점할 이미지 업로드.
input_dir = '/content/input_images'
os.makedirs(input_dir, exist_ok=True)

uploaded_images = files.upload()  # 추론할 이미지 upload. 여러 장은 한 번에 업로드
image_paths = []

for image_filename in uploaded_images.keys():
    new_path = os.path.join(input_dir, image_filename)
    shutil.move(image_filename, new_path)
    image_paths.append(new_path)

## 4. yolov5n으로 bbox 추론하기 (vscode에서는 subprocess 모듈을 사용하면 코드 내부에서 추론 가능)
!python detect.py --weights '{best_path}' --img 960 --conf 0.25 --source {input_dir} --save-txt --save-conf

# subprocess.run([
#     "python", "detect.py",
#     "--weights", "best.pt",
#     "--img", "640",
#     "--conf", "0.25",
#     "--source", "input_images",
#     "--save-txt",
#     "--save-conf"
# ], cwd="yolov5")

## 5.객관식과 단답형을 구분해서 ROI 추출 후 문제 번호로 정렬

# runs/detect 하위의 exp* 폴더 모두 찾기
exp_folders = glob.glob('/content/yolov5/runs/detect/exp*')
latest_exp = max(exp_folders, key=os.path.getmtime)

# 이미지 경로 설정
output_mc_dir = '/content/multiple_choice_roi'
output_sa_dir = '/content/short_answer_roi'
os.makedirs(output_mc_dir, exist_ok=True)
os.makedirs(output_sa_dir, exist_ok=True)

# 전체 문제 번호 카운트 (페이지 넘어도 번호 유지)
page_idx = 1

for image_path in sorted(image_paths):  # 정렬된 이미지 순서대로 진행
    image_filename = os.path.basename(image_path)
    image_basename = os.path.splitext(image_filename)[0]

    # 이미지 및 라벨 경로
    label_path = os.path.join(latest_exp, 'labels', f'{image_basename}.txt')
    if not os.path.exists(label_path):
        print(f"라벨 없음: {label_path}")
        continue

    # 이미지 로드
    image = cv2.imread(image_path)
    h, w = image.shape[:2]

    # 라벨 로딩 및 ROI 복원
    rois = []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            cls_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:5])
            cx_abs = x_center * w
            cy_abs = y_center * h
            x1 = int((x_center - width / 2) * w)
            y1 = int((y_center - height / 2) * h)
            x2 = int((x_center + width / 2) * w)
            y2 = int((y_center + height / 2) * h)
            rois.append({
                'class': cls_id,
                'cx': cx_abs,
                'cy': cy_abs,
                'coords': (x1, y1, x2, y2)
            })

    # 좌우 정렬 및 상하 정렬 (페이지마다 일관되게)
    rois_arr = np.array([[r['class'], r['cx'], r['cy']] for r in rois])
    x_threshold = np.mean(rois_arr[:, 1])
    left_col = [r for r in rois if r['cx'] < x_threshold]
    right_col = [r for r in rois if r['cx'] >= x_threshold]
    left_sorted = sorted(left_col, key=lambda r: r['cy'])
    right_sorted = sorted(right_col, key=lambda r: r['cy'])
    sorted_rois = left_sorted + right_sorted

    # ROI 저장
    for roi in sorted_rois:
        x1, y1, x2, y2 = roi['coords']
        roi_img = image[y1:y2, x1:x2]
        if roi['class'] == 0:  # 객관식
            qtype = "객관식"
            save_dir = output_mc_dir
        else:  # 단답형
            qtype = "단답형"
            save_dir = output_sa_dir

        roi_filename = f"[{qtype}_{page_idx}].jpg"
        roi_path = os.path.join(save_dir, roi_filename)
        cv2.imwrite(roi_path, roi_img)
        page_idx += 1

print(f"모든 이미지에서 ROI 총 {page_idx - 1}개 생성.")

Cloning into 'yolov5'...
remote: Enumerating objects: 17483, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 17483 (delta 77), reused 29 (delta 29), pack-reused 17378 (from 4)
Receiving objects: 100% (17483/17483), 16.35 MiB | 11.09 MiB/s, done.
Resolving deltas: 100% (11988/11988), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/

Saving best.pt to best.pt


Saving image_all_aug_075.jpg to image_all_aug_075.jpg
Saving image_all_aug_128.jpg to image_all_aug_128.jpg
Saving image_sa_all_aug_004.jpg to image_sa_all_aug_004.jpg
Saving image_sa_all_aug_010.jpg to image_sa_all_aug_010.jpg
Saving image_sa_bright+noise_032.jpg to image_sa_bright+noise_032.jpg
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
detect: weights=['/content/best.pt'], source=/content/input_images, data=data/coco128.yaml, imgsz=[960, 960], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=True, save_format=0, save_csv=False, save_conf=True, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist